# Working code for the biomechzoo Ensembler
This is the plotting library associated with the biomechzoo toolbox. This means that the data inputted into these plotting functions need to be _.zoo files_ or very similarly structured as nested dictionaries.

In [1]:
# import statements
from src.ensembler import Ensembler
from src.plot_spec import PlotSpec
# line plot renderers
from src.renderers import (IndividualLinesRenderer, MeanSDRenderer,
                           EventOverlayRenderer)
# event plot renderers
from src.renderers import (ViolinRenderer,
                           BlandAltmanRenderer, ScatterRenderer)
# combiner renderers
from src.renderers import CompositeRenderer
from src.helpers import ConditionSpec, ConditionSource

## Variables
The following variables should be set a priori for it to work
1. **fld**: path to the root folder where the data is stored
2. **ConditionSpec**: dictionary specifying how the conditions are defined (within **FOLDER** or **CHANNEL**)
3. **channels**: the channel names of interest
4. **str_match**: list of regular expression that matches the subject names. if emtpy; provide a subject list
5. **subj_list**: list of strings containing the subject names. if empty; provide a str_match.
6. **rows**: number of subplot rows
7. **cols**: number of subplot cols.

```python
condition_spec = ConditionSpec(
    source = ConditionSource.FOLDER or ConditionSource.CHANNEL,
    conditions= [list of conditions],
    channel_map = {condition: full channel name})
```

### Example of setting up the structure

In [2]:
# Set up variables.
fld = "/Users/Werk/Documents/Postdoc-McGill/areve-mcgill/data/Round2/16-qc"

spec = ConditionSpec(
    source     = ConditionSource.CHANNEL,
    conditions = ["vicon", "areve", 'pig'],
    channel_map = {
        "vicon": "RS_abduction_vicon",
        "areve": "RS_abduction_areve",
        "pig" : "RS_abduction_pig",
    }
)
channels = ['RS_abduction']
# str_match = [r"\b\d{3}[A-Z]{2}\b", r"\b\d{3}[A-Z]{3}\b"]
subj_list = ["014LP"]
events = ["max"]
rows = 1
cols = 3

### Example of individual line plots, events overlay and mean +/- standard deviation plots
Immediatly assigning it to a figure

In [3]:
lines_and_events = CompositeRenderer(IndividualLinesRenderer(), EventOverlayRenderer())
fig = (
    Ensembler(
        in_folder=fld, channels=channels,
        n_rows=rows, n_cols=cols,
        subj_list=subj_list, condition_spec=spec)
    .add_subplot(PlotSpec('RS_abduction', "vicon", row=1, col=1, renderer=IndividualLinesRenderer()))
    .add_subplot(PlotSpec('RS_abduction', "areve", row=1, col=1, renderer=IndividualLinesRenderer()))

    .add_subplot(PlotSpec('RS_abduction', "vicon", row=1, col=2, renderer=lines_and_events, events=events))
    .add_subplot(PlotSpec('RS_abduction', "areve", row=1, col=2, renderer=lines_and_events, events=events))

    .add_subplot(PlotSpec('RS_abduction', "vicon", row=1, col=3, renderer=MeanSDRenderer()))
    .add_subplot(PlotSpec('RS_abduction', "areve", row=1, col=3, renderer=MeanSDRenderer()))

    .build(title="Shoulder abduction - AReve vs Vicon vs Plug-in Gait")
)
fig.show()


Built up individually and with all subjects

In [4]:
str_match = [r"\b\d{3}[A-Z]{2}\b", r"\b\d{3}[A-Z]{3}\b"]

lines_and_events = CompositeRenderer(IndividualLinesRenderer(), EventOverlayRenderer())
ens = Ensembler( in_folder=fld, channels=channels,
                 n_rows=rows, n_cols=cols,
                 str_match=str_match, condition_spec=spec)
ens.add_subplot(PlotSpec('RS_abduction', "vicon",
                         row=1, col=1, renderer=IndividualLinesRenderer()))
ens.add_subplot(PlotSpec('RS_abduction', "areve",
                         row=1, col=1, renderer=IndividualLinesRenderer()))

ens.add_subplot(PlotSpec('RS_abduction', "vicon",
                         row=1, col=2, renderer=lines_and_events, events=events))
ens.add_subplot(PlotSpec('RS_abduction', "areve",
                         row=1, col=2, renderer=lines_and_events, events=events))

ens.add_subplot(PlotSpec('RS_abduction', "vicon",
                         row=1, col=3, renderer=MeanSDRenderer()))
ens.add_subplot(PlotSpec('RS_abduction', "areve",
                         row=1, col=3, renderer=MeanSDRenderer()))

fig = ens.build(title="Shoulder abduction - AReve vs Vicon vs Plug-in Gait")
fig.show()

## Extras
The returned fig from the ensembler allows the user all the functionality of plotly figures.
e.g. add annotations to the plot

In [5]:
fig.add_annotation(x=80, y=80, showarrow=False,
            text="RMSE: 24 deg",
            yshift=10, font=dict(size=18), row=1,col=3)

## Example violin plot.
Condition is now set on the FOLDER level.

In [6]:
# Set up variables
fld = "/Users/Werk/Documents/Postdoc-McGill/breast-reduction/data/stats"
spec = ConditionSpec(
    source = ConditionSource.FOLDER,
    conditions = ["Pre-surgery", "Post-surgery"]
)

channels = ['ax_pelvis_tilt_corr']
str_match = [r"\b\d{3}[A-Z]{2}\b", r"\b\d{3}[A-Z]{3}\b"]
events = ['impact_peak_mean', 'loading_rate_mean']
rows = 1
cols = 2


In [7]:
fig = (
    Ensembler(
        in_folder=fld,  channels=channels,
        n_rows=rows,  n_cols=cols,
        str_match=str_match,
        condition_spec=spec,
        events=events)
    .add_subplot(PlotSpec(
        channel="ax_pelvis_tilt_corr",
        condition = "Pre-surgery", companions = ["Post-surgery"],
        row=1, col=1,
        renderer=ViolinRenderer(),
        events=['impact_peak_mean'],)
    )
    .add_subplot(PlotSpec(
        channel="ax_pelvis_tilt_corr",
        condition="Pre-surgery", companions=["Post-surgery"],
        row=1, col=2,
        renderer=ViolinRenderer(),
        events=['loading_rate_mean'])
    )
    .build(title="Metrics pre- & post-BVRS")
)
fig.show()

## Example Scatter plot with regression line and Band-Altman plot
The show_subjects operator allows users to plot the individual subject colors in the markers. Might make it useful to figure out individual trends.

In [12]:
fld = "/Users/Werk/Documents/Postdoc-McGill/areve-mcgill/data/Round2/16-qc"

spec = ConditionSpec(
    source     = ConditionSource.CHANNEL,
    conditions = ["vicon", "areve", 'pig'],
    channel_map = {
        "vicon": "RS_abduction_vicon",
        "areve": "RS_abduction_areve",
        "pig" : "RS_abduction_pig",
    }
)
channels = ['RS_abduction']
str_match = [r"\b\d{3}[A-Z]{2}\b", r"\b\d{3}[A-Z]{3}\b"]
# subj_list = ["001CEJ"]
events = ["max"]
rows = 1
cols = 2

In [14]:
fig = (
    Ensembler(in_folder=fld, channels=channels, n_rows= rows, n_cols =cols, str_match=str_match, condition_spec=spec)
    .add_subplot(PlotSpec(channel='RS_abduction',
                          condition="areve", companions=["vicon"],
                          row=1, col=1,
                          renderer=ScatterRenderer(regression_line= True, show_subjects = True), events=events))
    .add_subplot(PlotSpec(channel='RS_abduction',
                          condition="areve", companions=["vicon"],
                          row=1, col=2, renderer=BlandAltmanRenderer(use_events= True, show_subjects=True), events=events))
    .build(title="Max Right Shoulder Abduction")
)

fig.show()